In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nb
    
sys.path.insert(1, "../../")

import precision_mapping as pm
import CAP_tools


sys.path.insert(1, "../../network_control")
import surface_mapping as sfm

ModuleNotFoundError: No module named 'mat73'

In [ ]:
exclude_subcortex = True
mask = False
sparsity = 0.1

silent = True

pm.utils.printer.silent = silent

In [ ]:
pm.functional

In [3]:
def load_FC_cortex(FC_path, template_cifti):
    """ """
    sc = scipy.sparse.load_npz(FC_path)
    cortex_index = pm.na.get_cortex_data(np.arange(sc.shape[0]).reshape(1, -1), template_cifti)[0]
    return sc[cortex_index][:, cortex_index]

def load_partition_labels(partition_path, template_cifti):
    """ """
    partition = np.load(partition_path)
    vertex_labels = pm.na.get_partition_cortex(partition, template_cifti)
    vertex_labels[np.isnan(vertex_labels)] = np.nanmax(vertex_labels) + 1
    remapped_vertex_labels = np.unique(vertex_labels, return_inverse=True)[1]
    return remapped_vertex_labels

def load_network_labels(network_path):
    return np.load(network_save_path)[0].astype(int)


def vertex_plot(values, template_cifti, ax=None, pclip=(None, None), **kwargs):
    """ """
    values = CAP_tools.utils.cifti_map(None, values, template_cifti)
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4))
    return sfm.surface_plot(values, ax=ax, **kwargs)

In [4]:
def check_multiple_args(args, main_dtype=str):
    """ """
    if any(not isinstance(arg, main_dtype) for arg in args):
        assert all(len(arg) == len(args[0]) for arg in args[1:]), "arg lists not same length"
        return True

    return False

# def multiple_arg_vectorize(func, *args, main_dtype=str, **kwargs):
#     """ """
#     if check_multiple_args(args, main_dtype=main_dtype):
#         np.vectorize(func)(*args, **kwargs)
#         return True

#     return False


def write_parcel_dlabel(dtseries_path, parcel_partition_path, parcel_dlabel_path, template_cifti, cmap=None):
    """ """
    # list of paths version
    args = [dtseries_path, parcel_partition_path, parcel_dlabel_path]
    if check_multiple_args(args, main_dtype=str):
        np.vectorize(write_parcel_dlabel)(*args, template_cifti=template_cifti, cmap=cmap)
        return

    vertex_parcel_labels = load_partition_labels(parcel_partition_path, template_cifti)
    
    parcel_values_int = CAP_tools.utils.cifti_map(None, vertex_parcel_labels, template_cifti)
    pm.plot.write_dlabel_precision_map(parcel_values_int, parcel_dlabel_path, cmap=cmap)


def write_network_dlabel(dtseries_path, network_partition_path, network_dlabel_path, template_cifti, cmap=None):
    """ """
    # list of paths version
    args = [dtseries_path, network_partition_path, network_dlabel_path]
    if check_multiple_args(args, main_dtype=str):
        np.vectorize(write_network_dlabel)(*args, template_cifti=template_cifti, cmap=cmap)
        return

    # single path version
    vertex_network_labels, vertex_network_strings = np.load(network_partition_path)
    vertex_network_labels = vertex_network_labels.astype(int)
    network_values_int = CAP_tools.utils.cifti_map(None, vertex_network_labels, template_cifti)
    network_name_to_num_map = {z: vertex_network_strings[np.where(vertex_network_labels == z)[0][0]]
                               for z in np.unique(vertex_network_labels)}
    network_dlabel_values = {
        "left": np.array(list(map(network_name_to_num_map.get, network_values_int["left"]))),
        "right": np.array(list(map(network_name_to_num_map.get, network_values_int["right"])))
    }
    pm.plot.write_dlabel_precision_map(network_dlabel_values, network_dlabel_path, cmap=cmap)


def parcel_plot(parcel_partition_path, network_partition_path, sample_label, save_path, template_cifti, close=True):

    args = [parcel_partition_path, network_partition_path, sample_label, save_path]
    if check_multiple_args(args, main_dtype=str):
        np.vectorize(parcel_plot)(*args, template_cifti=template_cifti, close=close)
        return

    vertex_parcel_labels = load_partition_labels(parcel_partition_path, template_cifti)
    vertex_network_labels, _ = np.load(network_partition_path)
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    fig.tight_layout(h_pad=2)
    ax, _ = vertex_plot(vertex_parcel_labels, template_cifti, cmap=plt.cm.Spectral, outline=False, ax=axes[0])
    ax.set_title(f"{sample_label} Assigned Parcels")
    
    ax, _ = vertex_plot(vertex_network_labels, template_cifti, cmap=plt.cm.Spectral, ax=axes[1])
    ax.set_title(f"{sample_label} Assigned Networks")
    
    fig.savefig(save_path, bbox_inches='tight')
    
    if close:
        plt.close()
    return

In [5]:
def create_pm_paths(subject_ids, sample_labels, precision_maps_out_dir):
    """ """
    vertex_fc_paths, parcel_partition_paths, network_partition_paths = [], [], []
    parcel_dlabel_paths, network_dlabel_paths, plot_save_paths = [], [], []

    for subject_id, sample_label in zip(subject_ids, sample_labels):
        subject_pm_dir = os.path.join(precision_maps_out_dir, subject_id)
        if not os.path.exists(subject_pm_dir):
            os.mkdir(subject_pm_dir)

        subject_generic_file_name = f"{subject_pm_dir}/{sample_label}_{{file_ending}}"
        
        vertex_fc_paths.append(subject_generic_file_name.format(file_ending="vertex_FC.npz"))
        parcel_partition_paths.append(subject_generic_file_name.format(file_ending="parcel_partition.npy"))
        network_partition_paths.append(subject_generic_file_name.format(file_ending="network_partition.npy"))

        parcel_dlabel_paths.append(subject_generic_file_name.format(file_ending="parcels.dlabel.nii"))
        network_dlabel_paths.append(subject_generic_file_name.format(file_ending="networks.dlabel.nii"))
        plot_save_paths.append(subject_generic_file_name.format(file_ending="_parcellation_plot.png"))

    return vertex_fc_paths, parcel_partition_paths, network_partition_paths, parcel_dlabel_paths, network_dlabel_paths, plot_save_paths

In [6]:
def run_precision_mapping(dtseries_paths, subject_ids, sample_labels, precision_maps_out_dir,
                          overwrite=False, silent=True, device="cpu"):
    """ """

    dtseries_paths = [dtseries_paths] if isinstance(dtseries_paths, str) else dtseries_paths
    subject_ids = [subject_ids] if isinstance(subject_ids, str) else subject_ids
    sample_labels = [sample_labels] if isinstance(sample_labels, str) else sample_labels
    
    path_sets = create_pm_paths(subject_ids, sample_labels, precision_maps_out_dir)
    vertex_fc_paths, parcel_partition_paths, network_partition_paths, _, _, _ = path_sets
    _, _, _, parcel_dlabel_paths, network_dlabel_paths, plot_save_paths = path_sets

    pm.functional_connectivity.generate_correlation_matrix(dtseries_paths, vertex_fc_paths, block_size=1000,
                                                                      overwrite=overwrite, backend="torch", device=device)
    pm.parcellate.parcel_detection(vertex_fc_paths, parcel_partition_paths, n_cores=1, n_reps=50, overwrite=overwrite, silent=silent)
    pm.na.assign_networks_batch(dtseries_paths, parcel_partition_paths, network_partition_paths, overwrite=overwrite)

    template_cifti = nb.load(dtseries_paths[0])

    write_parcel_dlabel(dtseries_paths, parcel_partition_paths, parcel_dlabel_paths, template_cifti)
    write_network_dlabel(dtseries_paths, network_partition_paths, network_dlabel_paths,
                         template_cifti, cmap=NETWORK_CMAP)
    parcel_plot(parcel_partition_paths, network_partition_paths, sample_labels, plot_save_paths, template_cifti)
    return path_sets

In [7]:
precision_maps_out_dir = "/System/Volumes/Data/data/data7/pfmsb_AAAV5170/analyses/precision_maps"

In [13]:
#changed by VL 11/3/
dtseries_path = "/data/data7/pfmsb_AAAV5170/analyses/precision_maps/sub-pfmsb001/bolddata/sub-pfmsb001_task-rest_space-fsLR_den-91k_desc-denoised_bold.dtseries.nii"

subject_id = "sub-pfmsb001"
sample_label = "sub-pfmsb001_ses-00105_task-rest_UNSmoothed_251119"

In [14]:
NETWORK_CMAP = {'Auditory': np.array([170, 83, 246, 255])/255,
'CinguloOpercular/Action-mode': np.array([70, 10, 146, 255])/255,
'Default_Anterolateral': np.array([145, 24, 103, 255])/255,
'Default_Dorsolateral': np.array([213, 156, 65, 255])/255,
'Default_Parietal': np.array([230, 53, 35, 255])/255,
'Default_Retrosplenial': np.array([255, 254, 217, 255])/255,
'DorsalAttention': np.array([96, 213, 60, 255])/255,
'Frontoparietal': np.array([254, 253, 84, 255])/255,
'MedialParietal': np.array([2, 87, 255, 255])/255,
'None': np.array([198, 198, 198, 255])/255,
'Premotor/DorsalAttentionII': np.array([253, 130, 255, 255])/255,
'Salience': np.array([0, 0, 0, 255])/255,
'SomatoCognitiveAction': np.array([120, 19, 20, 255])/255,
'Somatomotor_Face': np.array([254, 126, 0, 255])/255,
'Somatomotor_Foot': np.array([12, 80, 4, 255])/255,
'Somatomotor_Hand': np.array([113, 253, 254, 255])/255,
'Visual_Dorsal/VentralStream': np.array([47, 114, 180, 255])/255,
'Visual_Lateral': np.array([27, 14, 145, 255])/255,
'Visual_V1': np.array([181, 209, 140, 255])/255,
'Visual_V5': np.array([254, 180, 97, 255])/255,
'Language': np.array([62, 153, 153, 255])/255}

In [15]:
ps = run_precision_mapping(dtseries_path, subject_id, sample_label, precision_maps_out_dir, overwrite=True, silent=False)

Generating vertex-level FC:   0%|          | 0/1 [00:00<?, ?it/s]

SparseCorrelator Block Analysis:   0%|          | 0/4278.0 [00:00<?, ?it/s]

Running infomap parcel detection:   0%|          | 0/1 [00:00<?, ?it/s]

/System/Volumes/Data/data/data7/pfmsb_AAAV5170/analyses/precision_maps/sub-pfmsb001/sub-pfmsb001_ses-00105_task-rest_UNSmoothed_251119_vertex_FC.npz
  Infomap v2.8.0 starts at 2025-11-19 17:49:02
  -> Input network: 
  -> No file output!
  -> Configuration: two-level
                    seed = 42
                    num-trials = 50
  -> Ordinary network input, using the Map Equation for first order network flows
Calculating global network flow using flow model 'undirected'... 
  -> Using undirected links.
  => Sum node flow: 1, sum link flow: 1
Build internal network with 84551 nodes and 8332312 links...
  -> One-level codelength: 14.7400916

Trial 1/50 starting at 2025-11-19 17:49:04
Two-level compression: 19% 0.83% 0.0793210394% 0.0142218419% 
Partitioned to codelength 1.10278486 + 10.7768577 = 11.87964256 in 4246 modules.

=> Trial 1/50 finished in 5.0639695s with codelength 11.8796426

Trial 2/50 starting at 2025-11-19 17:49:09
Two-level compression: 19% 1% 0.0363885465% 0.04260347

  0%|          | 0/1 [00:00<?, ?it/s]

0.0
0.0
0.0


In [182]:
smooth_dtseries_path = "/System/Volumes/Data/data/data7/pfmsb_AAAV5170/patepfmsb/derivatives/xcp-0.10.0rc3/sub-pfmsb001/ses-00101/func/sub-pfmsb001_ses-00101_task-rest_acq-day1_run-01_space-fsLR_den-91k_desc-denoisedSmoothed_bold.dtseries.nii"

smooth_sample_label = "sub-pfmsb001_ses-00101_task-rest_acq-day1_run-01_smoothed"

ps = run_precision_mapping(smooth_dtseries_path, subject_id, smooth_sample_label,
                           precision_maps_out_dir, overwrite=False, silent=False)

Generating vertex-level FC:   0%|          | 0/1 [00:00<?, ?it/s]

Running infomap parcel detection:   0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

/Users/cole/miniconda3/envs/nct/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:2605: RuntimeWarning: divide by zero encountered in func (vectorized)
  outputs = ufunc(*inputs)


In [183]:
hcp_dtseries_path = "/data/data7/HCP_7T/100610/MNINonLinear/Results/rfMRI_REST_7T_ALL/rfMRI_REST_7T_ALL_Atlas_hp2000_clean_nilearn.dtseries.nii"
hcp_subject_id = "HCP-100610"
hcp_sample_label = "HCP-100610_rfMRI_REST_7T_ALL"

run_precision_mapping(hcp_dtseries_path, hcp_subject_id, hcp_sample_label, precision_maps_out_dir, overwrite=False, silent=False);

Generating vertex-level FC:   0%|          | 0/1 [00:00<?, ?it/s]

Running infomap parcel detection:   0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

/Users/cole/miniconda3/envs/nct/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:2605: RuntimeWarning: divide by zero encountered in func (vectorized)
  outputs = ufunc(*inputs)


# Plotting